# Real Usage With Pixi And Mojo

This notebook shows a complete Kayak flow on real text:

- encode query and document text to ColBERT-style 128-dimensional vectors
- build a reusable `LateIndex`
- run exact search on the Mojo backend
- run a search plan with approximate candidate generation and exact reranking
- run batch search
- run the clause-text verifier

The retrieval API used below is the public `import kayak` surface.

This notebook uses ColBERT's public Python API to generate example token vectors on CPU. If you already have token-level embeddings, skip that step and pass them directly to `kayak.query(...)` and `kayak.documents(...)`.

The first encoding call downloads the ColBERT checkpoint if it is not already cached.

In [1]:
import warnings

from colbert.infra.config import ColBERTConfig
from colbert.modeling.checkpoint import Checkpoint
import kayak
import torch

DEFAULT_MODEL_NAME = "colbert-ir/colbertv2.0"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="CUDA is not available.*", category=UserWarning)
warnings.filterwarnings(
    "ignore",
    message="torch.cuda.amp.GradScaler is enabled, but CUDA is not available.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="resource_tracker: There appear to be .* leaked semaphore objects.*",
    category=UserWarning,
)

torch.set_num_threads(1)
torch.multiprocessing.set_sharing_strategy("file_system")

checkpoint = Checkpoint(
    DEFAULT_MODEL_NAME,
    colbert_config=ColBERTConfig(gpus=0),
    verbose=0,
)

def encode_query_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.queryFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()


def encode_document_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.docFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()

BACKEND = kayak.MOJO_EXACT_CPU_BACKEND

print("available_backends:", kayak.available_backends())
print("backend_info:", kayak.backend_info(BACKEND))
print("encoder_model:", DEFAULT_MODEL_NAME)

/Users/teilomillet/Code/kayak/.pixi/envs/default/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


available_backends: ('numpy_reference', 'mojo_exact_cpu')
backend_info: BackendInfo(name='mojo_exact_cpu', available=True, requires_mojo=True, query_layouts=('nested', 'flat_dim128'), index_layouts=('packed', 'hybrid_flat_dim128'), availability_reason='Kayak can invoke Mojo via: /Users/teilomillet/Code/kayak/.pixi/envs/default/bin/mojo')
encoder_model: colbert-ir/colbertv2.0


In [2]:
documents = [
    {
        "doc_id": "apple_m2_ultra",
        "text": "Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.",
    },
    {
        "doc_id": "nvidia_h100",
        "text": "NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.",
    },
    {
        "doc_id": "modular_mojo",
        "text": "Modular develops the Mojo programming language and compiler tools for AI systems.",
    },
    {
        "doc_id": "colbert_late_interaction",
        "text": "ColBERT uses late interaction with token-level MaxSim scoring instead of a single document embedding.",
    },
    {
        "doc_id": "pixi_env_manager",
        "text": "Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.",
    },
    {
        "doc_id": "lancedb_vectors",
        "text": "LanceDB stores vectors on disk and supports similarity search workloads.",
    },
]

queries = [
    "which company makes the M2 Ultra chip?",
    "what language does Modular build?",
    "which retrieval model uses late interaction?",
    "what tool installs Python and Mojo together?",
]

documents

[{'doc_id': 'apple_m2_ultra',
  'text': 'Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.'},
 {'doc_id': 'nvidia_h100',
  'text': 'NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.'},
 {'doc_id': 'modular_mojo',
  'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'},
 {'doc_id': 'colbert_late_interaction',
  'text': 'ColBERT uses late interaction with token-level MaxSim scoring instead of a single document embedding.'},
 {'doc_id': 'pixi_env_manager',
  'text': 'Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.'},
 {'doc_id': 'lancedb_vectors',
  'text': 'LanceDB stores vectors on disk and supports similarity search workloads.'}]

In [3]:
encoded_documents = [encode_document_text(doc["text"]) for doc in documents]
encoded_queries = {text: encode_query_text(text) for text in queries}

print("document_vector_counts:", [len(vectors) for vectors in encoded_documents])
print("query_vector_counts:", {text: len(vectors) for text, vectors in encoded_queries.items()})

[Apr 15, 09:33:57] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


document_vector_counts: [23, 24, 16, 24, 28, 17]
query_vector_counts: {'which company makes the M2 Ultra chip?': 32, 'what language does Modular build?': 32, 'which retrieval model uses late interaction?': 32, 'what tool installs Python and Mojo together?': 32}


In [4]:
index = kayak.documents(
    [doc["doc_id"] for doc in documents],
    encoded_documents,
    texts=[doc["text"] for doc in documents],
).pack()

documents_by_id = {doc["doc_id"]: doc["text"] for doc in documents}

def describe_hits(hits):
    return [
        {
            "doc_id": hit.doc_id,
            "score": round(hit.score, 3),
            "text": documents_by_id[hit.doc_id],
        }
        for hit in hits
    ]

print("layout:", index.layout)
print("document_count:", index.document_count)
print("total_vector_count:", index.total_vector_count)
print("vector_dim:", index.vector_dim)

layout: packed
document_count: 6
total_vector_count: 132
vector_dim: 128


## Exact Search For One Query

In [5]:
query_text = "which company makes the M2 Ultra chip?"
query = kayak.query(encoded_queries[query_text], text=query_text)

scores = kayak.maxsim(query, index, backend=BACKEND)
hits = kayak.search(query, index, k=3, backend=BACKEND)

print("top_scores:")
describe_hits(scores.topk(3))

top_scores:


[{'doc_id': 'apple_m2_ultra',
  'score': 26.585,
  'text': 'Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.'},
 {'doc_id': 'nvidia_h100',
  'score': 9.421,
  'text': 'NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.'},
 {'doc_id': 'modular_mojo',
  'score': 9.229,
  'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'}]

## Explicit 128-Dim Flat And Hybrid Layouts

In [6]:
flat_query = query.to_layout("flat_dim128")
hybrid_index = index.to_layout("hybrid_flat_dim128")

flat_scores = kayak.maxsim(flat_query, hybrid_index, backend=BACKEND)
describe_hits(flat_scores.topk(3))

[{'doc_id': 'apple_m2_ultra',
  'score': 26.585,
  'text': 'Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.'},
 {'doc_id': 'nvidia_h100',
  'score': 9.421,
  'text': 'NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.'},
 {'doc_id': 'modular_mojo',
  'score': 9.229,
  'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'}]

## Approximate Candidate Generation Plus Exact Reranking

In [7]:
plan = kayak.document_proxy_search_plan(
    final_k=2,
    candidate_k=4,
    query_vector_budget=16,
    document_vector_budget=16,
)

plan_result = kayak.search_with_plan(query, index, plan, backend=BACKEND)

print("candidate_hits:")
print(describe_hits(plan_result.candidate_stage.hits))

print("final_hits:")
print(describe_hits(plan_result.hits))

print("candidate_profile:", plan_result.candidate_stage.profile)
print("stage2_profile:", plan_result.stage2)

candidate_hits:
[{'doc_id': 'apple_m2_ultra', 'score': 0.232, 'text': 'Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.'}, {'doc_id': 'modular_mojo', 'score': 0.136, 'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'}, {'doc_id': 'nvidia_h100', 'score': 0.08, 'text': 'NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.'}, {'doc_id': 'pixi_env_manager', 'score': 0.058, 'text': 'Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.'}]
final_hits:
[{'doc_id': 'apple_m2_ultra', 'score': 26.585, 'text': 'Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.'}, {'doc_id': 'nvidia_h100', 'score': 9.421, 'text': 'NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.'}]
candidate_profile: SearchStageProfile(stage_name='document_proxy', input_hit_count=6, output_hit_count=4, query_vect

## Batch Search Against The Same Index

In [8]:
batch = kayak.query_batch([encoded_queries[text] for text in queries])
hits_by_query = kayak.search_batch(batch, index, k=2, backend=BACKEND)

batch_results = {
    query_text: describe_hits(hits)
    for query_text, hits in zip(queries, hits_by_query, strict=True)
}

batch_results

{'which company makes the M2 Ultra chip?': [{'doc_id': 'apple_m2_ultra',
   'score': 26.585,
   'text': 'Apple announced the M2 Ultra chip for high-end Macs and workstation-class desktops.'},
  {'doc_id': 'nvidia_h100',
   'score': 9.421,
   'text': 'NVIDIA introduced the H100 GPU for large-scale AI training and inference workloads.'}],
 'what language does Modular build?': [{'doc_id': 'modular_mojo',
   'score': 25.309,
   'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'},
  {'doc_id': 'pixi_env_manager',
   'score': 9.547,
   'text': 'Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.'}],
 'which retrieval model uses late interaction?': [{'doc_id': 'colbert_late_interaction',
   'score': 18.514,
   'text': 'ColBERT uses late interaction with token-level MaxSim scoring instead of a single document embedding.'},
  {'doc_id': 'lancedb_vectors',
   'score': 8.276,
   'text': 'LanceDB stores vector

## Clause-Text Verification

In [9]:
verifier_query_text = "which company develops the Mojo programming language?"
verifier_query = kayak.query(
    encode_query_text(verifier_query_text),
    text=verifier_query_text,
)

verifier_plan = kayak.exact_full_scan_search_plan(
    final_k=2,
    candidate_k=4,
    stage3_verifier=kayak.clause_text_stage3_verifier_operator(),
)

verifier_result = kayak.search_with_plan(
    verifier_query,
    index,
    verifier_plan,
    backend=BACKEND,
)

print("stage3_hits:")
print(describe_hits(verifier_result.hits))
print("stage3_profile:", verifier_result.stage3_verifier)

stage3_hits:
[{'doc_id': 'modular_mojo', 'score': 28.955, 'text': 'Modular develops the Mojo programming language and compiler tools for AI systems.'}, {'doc_id': 'pixi_env_manager', 'score': 18.339, 'text': 'Pixi can create a Python 3.11 environment, install Mojo, and add PyPI packages in the same project.'}]
stage3_profile: SearchStageProfile(stage_name='clause_text', input_hit_count=4, output_hit_count=2, query_vector_count=0, document_count=4, document_vector_count=0, document_text_count=4, materialized_artifacts=(StageArtifactMaterialization(family='document_text', document_count=4, document_vector_count=0, document_text_count=4),))
